# Triplet Dataset Setup

This notebook loads one representative series from each dataset and standardizes them into a common structure for downstream benchmarking.

Datasets:

- `M5 / Walmart`
- `Favorita`
- `Amazon`

In [ ]:
from pathlib import Path
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'src').exists() and (cur / 'reports').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)

In [ ]:
from src.data_loaders.load_m5 import load_m5_single_series
from src.data_loaders.load_favorita import load_favorita_series
from src.data_loaders.load_amazon import load_amazon_series

In [ ]:
m5_df = load_m5_single_series(base_path='data/raw/m5', random_pick=True, seed=42)
favorita_df = load_favorita_series(base_path='data/raw/favorita', store_nbr=1, family='CLEANING')
amazon_df = load_amazon_series(
    base_path='data/raw/amazon_2023/review_categories',
    filename='Health_and_Household.jsonl.gz',
    top_rank=1,
    max_rows=100000,
)

series_map = {
    'm5': m5_df.copy(),
    'favorita': favorita_df.copy(),
    'amazon': amazon_df.copy(),
}

for name, df in series_map.items():
    print(name, df.shape, list(df.columns))

In [ ]:
def standardize_series(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'])
    out['sales'] = pd.to_numeric(out['sales'], errors='coerce').fillna(0.0)
    if 'price' not in out.columns:
        out['price'] = pd.NA
    out['dataset'] = dataset_name
    return out[['dataset', 'date', 'sales', 'price']].sort_values('date').reset_index(drop=True)

standardized = {name: standardize_series(df, name) for name, df in series_map.items()}
pd.concat(standardized.values(), ignore_index=True).head()